# Axis-convention diagnostic — which array axis is which coordinate?

**Purpose.** `ansatz.py` assumes axis 1 (rows) $\to x$ with pitch 50 um, axis 2 (cols) $\to y$ with
pitch 12.5 um. `AxisFixExpert` in the training notebook contradicts that for the *position* slots
only, by swapping them after the fact. But the same two axes also carry the **pitches**, both
**cluster-width angle magnitudes**, the **Lorentz `dy` correction**, and the **sign observables**.
If the axes are transposed, all five are wrong together and a two-slot swap fixes one of them.

This notebook measures the convention directly from data. Nothing is trained, nothing is written.

**The four measurements**

1. **Positions.** Regress label $x$ and label $y$ on the row-index centroid and the col-index
   centroid. Slopes are in *label-units per pixel*. The two pitches differ by exactly
   $50/12.5 = 4$, so the **ratio** of the two diagonal slopes is the decisive number and it does
   not depend on the label units being um.
2. **Widths.** Regress $|\cot\alpha|$, $|\cot\beta|$ on (row width $-1$) and (col width $-1$).
   The ansatz claims slope $= p/T$, i.e. $0.50$ for the 50 um axis and $0.125$ for the 12.5 um axis.
3. **Drifts.** Balanced accuracy of the between-slice centroid drift as a sign predictor, all four
   pairings. `ansatz.py` calls its mapping "cross-coupled"; if the axes are transposed it is
   actually the natural uncrossed pairing.
4. **Verdict.** Printed as a block at the end.

Run cells 1 -> 2 -> 3 in order. Cell 3 is the diagnostic and only needs `vg` and `labels_scale`
from cell 2, so you can re-run it alone.

In [1]:
# ---- 1. setup ----
import os
WORKDIR = "/depot/cms/private/users/kuang14/Smart_Pixel/smart-pixels-digitization"
HELPERS = os.path.join(WORKDIR, "two_bit_optimization_helpers")
os.chdir(WORKDIR)

import sys, json
sys.path.insert(0, HELPERS)

import numpy as np
import tensorflow as tf

for g in tf.config.list_physical_devices("GPU"):
    try: tf.config.experimental.set_memory_growth(g, True)
    except Exception: pass

from prepare_tfrecords import generate_tfrecords, load_tfrecords

print("TF", tf.__version__, "| workdir:", os.getcwd())
print("GPUs:", tf.config.list_physical_devices("GPU"))

2026-07-30 01:19:46.047166: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-07-30 01:19:46.047240: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-07-30 01:19:46.054367: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-07-30 01:19:46.121409: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-07-30 01:19:52.736129: W tensorflow/compiler/tf2

TF 2.15.1 | workdir: /depot/cms/private/users/kuang14/Smart_Pixel/smart-pixels-digitization
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
# ---- 2. data (analog, clean -- same config as train_symbolic_nexp_ablation) ----
import os, json, numpy as np
from prepare_tfrecords import generate_tfrecords, load_tfrecords

SEED             = 42
DATASET_DIR      = "/depot/cms/users/kuang14/Smart_Pixel/dataset_s_series/dataset_3sr/dataset_3sr_16x16_50x12P5_centeredIncidence_parquets"
SELECT_CONTAINED = True
TIMESLICES       = 2
BATCH            = 5000
TFRECORDS_EXIST  = True

_, _, tfr_tr, tfr_val = generate_tfrecords(
    dataset_dir=DATASET_DIR, model_type="ViT_Max",
    train_batch_size=BATCH, val_batch_size=BATCH,
    select_contained=SELECT_CONTAINED, timeslices=TIMESLICES,
    tfrecords_exist=TFRECORDS_EXIST, seed=SEED,
)
tg, vg = load_tfrecords(tfr_tr, tfr_val, noise=-1, digitize=False, seed=SEED)

labels_scale = json.load(open(os.path.join(tfr_tr, "metadata.json")))["labels_scale"]
print("labels_scale (x, y, cotA, cotB):", labels_scale)
print("val batches:", len(vg))

xb, yb = vg[0]
print("x:", np.asarray(xb).shape, "| y:", np.asarray(yb).shape)

Loading metadata from /depot/cms/users/kuang14/Smart_Pixel/dataset_s_series/dataset_3sr/dataset_3sr_16x16_50x12P5_centeredIncidence_parquets/TFR_files/2t/TFR_train_contained/metadata.json
Loading metadata from /depot/cms/users/kuang14/Smart_Pixel/dataset_s_series/dataset_3sr/dataset_3sr_16x16_50x12P5_centeredIncidence_parquets/TFR_files/2t/TFR_test_contained/metadata.json


Loading metadata from /depot/cms/users/kuang14/Smart_Pixel/dataset_s_series/dataset_3sr/dataset_3sr_16x16_50x12P5_centeredIncidence_parquets/TFR_files/2t/TFR_train_contained/metadata.json


Loading metadata from /depot/cms/users/kuang14/Smart_Pixel/dataset_s_series/dataset_3sr/dataset_3sr_16x16_50x12P5_centeredIncidence_parquets/TFR_files/2t/TFR_test_contained/metadata.json


labels_scale (x, y, cotA, cotB): [123.41016201181557, 30.929849811025303, 6.577498885094723, 1.9295648020239338]
val batches: 22


2026-07-30 01:20:13.027828: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3234 MB memory:  -> device: 0, name: NVIDIA A100-PCIE-40GB MIG 1g.5gb, pci bus id: 0000:81:00.0, compute capability: 8.0


x: (5000, 16, 16, 2) | y: (5000, 4)


In [3]:
# ---- 3. THE DIAGNOSTIC ----  needs only `vg` and `labels_scale` from cell 2
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score

N_BATCHES = 10          # 10 x 5000 = 50k events; raise for tighter errors
T_SENSOR  = 100.0       # um, sensor thickness (GEOMETRY['T'])

# ---------------- collect ----------------
nb = min(N_BATCHES, len(vg))
X = np.concatenate([np.asarray(vg[i][0], "float32") for i in range(nb)])   # (N,16,16,2)
Y = np.concatenate([np.asarray(vg[i][1], "float32") for i in range(nb)])   # (N,4) scaled
Yp = Y * np.asarray(labels_scale, "float32")                              # un-scaled labels
lab_x, lab_y, cotA, cotB = Yp[:, 0], Yp[:, 1], Yp[:, 2], Yp[:, 3]
print(f"events: {len(X)}")

# ------------- pixel-INDEX observables (no pitch assumed anywhere) -------------
idx = np.arange(16, dtype=np.float32)

def profiles(q3):                      # q3: (N,16,16) -> (row-profile, col-profile)
    return q3.sum(2), q3.sum(1)        # axis1-indexed, axis2-indexed

def centroid(prof):
    return (prof * idx).sum(1) / (prof.sum(1) + 1e-9)

qsum = np.maximum(X, 0.0).sum(-1)      # time-summed, relu'd (analog pedestals < 0)
prof_r, prof_c = profiles(qsum)
R, C = centroid(prof_r), centroid(prof_c)          # ROW-index and COL-index centroids
w_r  = (prof_r > 0).sum(1).astype(np.float32)      # row width  (pixels)
w_c  = (prof_c > 0).sum(1).astype(np.float32)      # col width  (pixels)

def slope_corr(f, lab):
    f = f - f.mean(); l = lab - lab.mean()
    s = float((f * l).sum() / ((f * f).sum() + 1e-12))
    c = float((f * l).sum() / (np.sqrt((f * f).sum() * (l * l).sum()) + 1e-12))
    return s, c

# ---------------- 1. POSITIONS ----------------
print("\n" + "=" * 74)
print("1. POSITIONS   slope = label-units per pixel of centroid movement")
print("=" * 74)
print(f"{'':22s}{'slope':>12s}{'corr':>10s}")
pos = {}
for fname, f in (("ROW centroid", R), ("COL centroid", C)):
    for lname, lab in (("label x", lab_x), ("label y", lab_y)):
        s, c = slope_corr(f, lab)
        pos[(fname[:3], lname[-1])] = (s, c)
        print(f"  {fname} -> {lname:8s}{s:12.3f}{c:10.3f}")

# which centroid owns which label (by |corr|)
row_owns = "x" if abs(pos[("ROW", "x")][1]) > abs(pos[("ROW", "y")][1]) else "y"
col_owns = "x" if abs(pos[("COL", "x")][1]) > abs(pos[("COL", "y")][1]) else "y"
s_row = abs(pos[("ROW", row_owns)][0])
s_col = abs(pos[("COL", col_owns)][0])
ratio = max(s_row, s_col) / (min(s_row, s_col) + 1e-12)
coarse = "ROW" if s_row > s_col else "COL"
print(f"\n  row centroid tracks label {row_owns};  col centroid tracks label {col_owns}")
print(f"  |slope| ratio coarse/fine = {ratio:.2f}   (expect 4.00 for 50 um / 12.5 um)")
print(f"  -> the 50 um (coarse) axis is {coarse}")

# ---------------- 2. WIDTHS ----------------
print("\n" + "=" * 74)
print("2. WIDTHS   ansatz claims |cot| = (width-1) * pitch / T  ->  slope = pitch/100")
print("=" * 74)
print(f"{'':30s}{'slope':>10s}{'corr':>10s}{'implied pitch (um)':>22s}")
wid = {}
for wname, w in (("ROW width-1", w_r - 1.0), ("COL width-1", w_c - 1.0)):
    for aname, a in (("|cotA|", np.abs(cotA)), ("|cotB|", np.abs(cotB))):
        s, c = slope_corr(w, a)
        wid[(wname[:3], aname)] = (s, c)
        print(f"  {wname} -> {aname:16s}{s:10.4f}{c:10.3f}{s * T_SENSOR:22.2f}")

a_from = "ROW" if abs(wid[("ROW", "|cotA|")][1]) > abs(wid[("COL", "|cotA|")][1]) else "COL"
b_from = "ROW" if abs(wid[("ROW", "|cotB|")][1]) > abs(wid[("COL", "|cotB|")][1]) else "COL"
print(f"\n  |cotA| is carried by the {a_from} width;  |cotB| by the {b_from} width")

# ---------------- 3. DRIFTS -> SIGN ----------------
print("\n" + "=" * 74)
print("3. DRIFTS   between-slice centroid drift as a sign predictor (balanced acc)")
print("=" * 74)
q0 = np.maximum(X[..., 0],  0.0)
q1 = np.maximum(X[..., -1], 0.0)
pr0, pc0 = profiles(q0)
pr1, pc1 = profiles(q1)
dR = centroid(pr1) - centroid(pr0)      # row-index drift
dC = centroid(pc1) - centroid(pc0)      # col-index drift
dR = np.nan_to_num(np.clip(dR, -16.0, 16.0))
dC = np.nan_to_num(np.clip(dC, -16.0, 16.0))

print(f"{'':26s}{'bal-acc':>10s}")
sgn = {}
for dname, d in (("ROW drift", dR), ("COL drift", dC)):
    for aname, a in (("sign(cotA)", cotA), ("sign(cotB)", cotB)):
        t = (a > 0).astype(int)
        clf = LogisticRegression(max_iter=5000).fit(d.reshape(-1, 1), t)
        ba = balanced_accuracy_score(t, clf.predict(d.reshape(-1, 1)))
        sgn[(dname[:3], aname)] = ba
        print(f"  {dname} -> {aname:14s}{ba:10.4f}")

sA = "ROW" if sgn[("ROW", "sign(cotA)")] > sgn[("COL", "sign(cotA)")] else "COL"
sB = "ROW" if sgn[("ROW", "sign(cotB)")] > sgn[("COL", "sign(cotB)")] else "COL"
print(f"\n  sign(cotA) comes from the {sA} drift;  sign(cotB) from the {sB} drift")

# ---------------- 4. VERDICT ----------------
print("\n" + "=" * 74)
print("4. VERDICT")
print("=" * 74)
print(f"  measured:  ROW(axis 1) = label {row_owns} , 50um axis = {coarse}")
print(f"             |cotA| <- {a_from} width , |cotB| <- {b_from} width")
print(f"             sign(cotA) <- {sA} drift , sign(cotB) <- {sB} drift")
print("  ansatz.py: ROW(axis 1) = label x , 50um axis = ROW")
print("             |cotA| <- ROW width , |cotB| <- COL width")
print("             sign(cotA) <- COL drift , sign(cotB) <- ROW drift")

transposed = (row_owns == "y")
if transposed:
    print("\n  => AXES ARE TRANSPOSED throughout the ansatz.")
    print("     Fix at the top of PhysicsAnsatz.call (one transpose), NOT by permuting")
    print("     output slots. AxisFixExpert then becomes unnecessary.")
else:
    print("\n  => positions are NOT transposed; AxisFixExpert is doing something else.")
    print("     STOP and re-read before editing ansatz.py.")

consistent = (a_from == ("COL" if transposed else "ROW"))
print(f"  angle branch consistent with the position verdict: {consistent}")
if abs(ratio - 4.0) > 0.8:
    print(f"  !! slope ratio {ratio:.2f} is far from 4.00 -- labels may not be in um,")
    print("     or x/y are not both position labels. Check labels_scale before trusting.")

events: 50000

1. POSITIONS   slope = label-units per pixel of centroid movement
                             slope      corr
  ROW centroid -> label x        0.103     0.002
  ROW centroid -> label y       10.462     0.914
  COL centroid -> label x       43.226     0.927
  COL centroid -> label y       -0.025    -0.002

  row centroid tracks label y;  col centroid tracks label x
  |slope| ratio coarse/fine = 4.13   (expect 4.00 for 50 um / 12.5 um)
  -> the 50 um (coarse) axis is COL

2. WIDTHS   ansatz claims |cot| = (width-1) * pitch / T  ->  slope = pitch/100
                                   slope      corr    implied pitch (um)
  ROW width-1 -> |cotA|              0.0107     0.019                  1.07
  ROW width-1 -> |cotB|              0.1483     0.944                 14.83
  COL width-1 -> |cotA|              0.5189     0.990                 51.89
  COL width-1 -> |cotB|             -0.0022    -0.015                 -0.22

  |cotA| is carried by the COL width;  |cotB| by the